- `nchwToNhwc` kernel in eager takes 974µs(nearly 10% of total time) and not in compiled mode. `torch.compile` eliminated the memory layout conversion by keeping tensors in NHWC format
- `BatchNorm` is also fused in compile mode
- `compiled model` is slower in CPU

In [1]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import time
import json
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Callable

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

CONFIG = {
    'resolution'    : 384,
    'warmup_runs'   : 20,    
    'measure_runs'  : 100,
    'device'        : 'cuda' if torch.cuda.is_available() else 'cpu',
    'image_path'    : 'Images/people.jpg',
    'results_dir'   : Path('results'),
}
CONFIG['results_dir'].mkdir(exist_ok=True)

DEVICE = CONFIG['device']
RES    = CONFIG['resolution']
print(f'Device: {DEVICE}')
print(f'CUDA version: {torch.version.cuda}')
print(f'PyTorch version: {torch.__version__}')

Device: cuda
CUDA version: 12.8
PyTorch version: 2.10.0+cu128


In [2]:
@dataclass
class BenchmarkResult:
    name        : str
    mean_ms     : float
    p50_ms      : float
    p95_ms      : float
    p99_ms      : float
    fps         : float
    precision   : str

    def __str__(self):
        return (
            f"[{self.name}]\n"
            f"  Mean: {self.mean_ms:.2f}ms | P50: {self.p50_ms:.2f}ms | "
            f"P95: {self.p95_ms:.2f}ms | P99: {self.p99_ms:.2f}ms\n"
            f"  FPS: {self.fps:.1f} "
        )


def benchmark(name: str,
              infer_fn: Callable,
              input_tensor: torch.Tensor,
              warmup: int = CONFIG['warmup_runs'],
              runs: int = CONFIG['measure_runs'],
              precision: str = 'fp32') -> BenchmarkResult:
    """
    Benchmark an inference function using CUDA Events.

    CUDA Events explained:
    - start.record() stamps a timestamp into the GPU command queue
    - end.record() stamps another timestamp after all kernels finish
    - elapsed_time() reads the difference — pure GPU time, no CPU overhead
    """
    #cuDNN pick optimal algorithms & fill caches
    for _ in range(warmup):
        _ = infer_fn(input_tensor)
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    latencies = []

    for _ in range(runs):
        start_event = torch.cuda.Event(enable_timing=True)
        end_event   = torch.cuda.Event(enable_timing=True)

        start_event.record()
        _ = infer_fn(input_tensor)
        end_event.record()
        torch.cuda.synchronize()
        latencies.append(start_event.elapsed_time(end_event))

    arr         = np.array(latencies)
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)

    return BenchmarkResult(
        name        = name,
        mean_ms     = float(arr.mean()),
        p50_ms      = float(np.percentile(arr, 50)),
        p95_ms      = float(np.percentile(arr, 95)),
        p99_ms      = float(np.percentile(arr, 99)),
        fps         = 1000.0 / float(arr.mean()),
        precision   = precision,
    )

all_results: List[BenchmarkResult] = []

In [3]:
def preprocess(image_path: str,
               resolution: int = RES,
               device: str = DEVICE,
               dtype: torch.dtype = torch.float32,
               use_channels_last: bool = False) -> torch.Tensor:
    """
    Preprocess image into a model-ready tensor.
    use_channels_last: store in NHWC layout for CNN efficiency.
    """
    frame = cv2.imread(image_path)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (resolution, resolution))
    frame = frame.astype(np.float32) / 255.0

    tensor = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0)
    tensor = tensor.to(device=device, dtype=dtype)

    if use_channels_last:
        # .contiguous(memory_format=...) the data is actually reordered in memory
        tensor = tensor.contiguous(memory_format=torch.channels_last)

    return tensor


def save_depth_image(depth_tensor: torch.Tensor, path: str, label: str = ''):
    depth = depth_tensor.squeeze().detach().cpu().float().numpy()
    dmin, dmax = depth.min(), depth.max()
    depth_norm  = (depth - dmin) / (dmax - dmin + 1e-6)
    depth_uint8 = (depth_norm * 255).astype(np.uint8)
    colormap    = cv2.applyColorMap(depth_uint8, cv2.COLORMAP_INFERNO)
    if label:
        cv2.putText(colormap, label, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.imwrite(path, colormap)

input_fp32 = preprocess(CONFIG['image_path'], dtype=torch.float32)
input_fp16  = preprocess(CONFIG['image_path'], dtype=torch.float16)
input_fp32_nhwc = preprocess(CONFIG['image_path'], dtype=torch.float32,use_channels_last=True)
input_fp16_nhwc = preprocess(CONFIG['image_path'], dtype=torch.float16,use_channels_last=True)

In [4]:
model = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model = model.to(DEVICE).eval()

# Baseline FP32 ──
def infer_fp32(x):
    with torch.no_grad():
        return model(x)

result_baseline = benchmark('1_baseline_fp32', infer_fp32, input_fp32, precision='fp32')
all_results.append(result_baseline)
print(result_baseline)

with torch.no_grad():
    reference_output = model(input_fp32).detach().clone()
save_depth_image(reference_output, str(CONFIG['results_dir'] / '1_baseline_fp32.png'), 'FP32 Baseline')

Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


[1_baseline_fp32]
  Mean: 32.84ms | P50: 32.54ms | P95: 33.59ms | P99: 35.60ms
  FPS: 30.4 


In [5]:
def measure_accuracy_degradation(output_fp32: torch.Tensor,
                                  output_other: torch.Tensor,
                                  name: str):
    """
    AbsRel: mean absolute relative error.
    We use FP32 output as pseudo ground-truth (no real GT depth available here).
    In production you'd use a calibration dataset with real depth maps.
    """
    ref  = output_fp32.squeeze().float().cpu().numpy()
    pred = output_other.squeeze().float().cpu().numpy()
    mask    = ref > 0.01
    absrel  = np.mean(np.abs(pred[mask] - ref[mask]) / (ref[mask] + 1e-6))
    ref_n  = (ref - ref.min()) / (ref.max() - ref.min() + 1e-6)
    pred_n = (pred - pred.min()) / (pred.max() - pred.min() + 1e-6)
    struct_err = np.mean(np.abs(ref_n - pred_n))

    print(f'  [{name}] AbsRel: {absrel:.4f} | Structural error: {struct_err:.4f}')
    return absrel, struct_err

# FP16 (model.half())
model_fp16 = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model_fp16 = model_fp16.to(DEVICE).eval().half()

def infer_fp16(x):
    with torch.no_grad():
        return model_fp16(x)

result_fp16 = benchmark('2a_fp16', infer_fp16, input_fp16, precision='fp16')
all_results.append(result_fp16)
print(result_fp16)

with torch.no_grad():
    out_fp16 = model_fp16(input_fp16).detach().clone()
measure_accuracy_degradation(reference_output, out_fp16, 'fp16')
save_depth_image(out_fp16, str(CONFIG['results_dir'] / '2a_fp16.png'), 'FP16')

model_base = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model_base = model_base.to(DEVICE).eval()

def infer_amp(x):
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.float16):
            return model_base(x.float())

result_amp = benchmark('2b_amp_autocast', infer_amp, input_fp32, precision='amp')
all_results.append(result_amp)
print(result_amp)

Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


[2a_fp16]
  Mean: 18.65ms | P50: 20.05ms | P95: 21.54ms | P99: 22.00ms
  FPS: 53.6 
  [fp16] AbsRel: 0.0007 | Structural error: 0.0002


Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


[2b_amp_autocast]
  Mean: 19.18ms | P50: 19.15ms | P95: 19.49ms | P99: 19.67ms
  FPS: 52.1 


In [6]:
# ── Stage 3: FP16 + channels_last ──
model_nhwc = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model_nhwc = model_nhwc.to(DEVICE).eval().half()

# Convert model weights to channels_last layout cuDNN to use NHWC-optimized conv kernels throughout
model_nhwc = model_nhwc.to(memory_format=torch.channels_last)

def infer_fp16_nhwc(x):
    with torch.no_grad():
        return model_nhwc(x)

result_nhwc = benchmark('3_fp16_channels_last', infer_fp16_nhwc,
                         input_fp16_nhwc, precision='fp16')
all_results.append(result_nhwc)
print(result_nhwc)

with torch.no_grad():
    out_nhwc = model_nhwc(input_fp16_nhwc).detach().clone()
measure_accuracy_degradation(reference_output, out_nhwc, 'fp16+nhwc')

speedup = result_fp16.mean_ms / result_nhwc.mean_ms
print(f'\nchannels_last speedup over fp16: {speedup:.2f}x')

Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


[3_fp16_channels_last]
  Mean: 15.45ms | P50: 15.55ms | P95: 16.26ms | P99: 16.36ms
  FPS: 64.7 
  [fp16+nhwc] AbsRel: 0.0007 | Structural error: 0.0002

channels_last speedup over fp16: 1.21x


In [7]:
from torch.nn.utils.fusion import fuse_conv_bn_eval

def fuse_model_bn(model: nn.Module) -> nn.Module:
    """
    Fuse all Conv2d+BatchNorm2d pairs in the model.
      Conv:  y = W * x + b
      BN:    z = gamma * (y - mean) / sqrt(var + eps) + beta
      Merge: z = W_fused * x + b_fused   where:
             W_fused = W  * (gamma / std)          [per output channel]
             b_fused = (b - mean) * (gamma / std) + beta
      BN becomes nn.Identity() — zero runtime cost.
    """
    _fuse_bn_recursive(model)
    return model


def _fuse_bn_recursive(parent: nn.Module) -> None:
    """
    Scan each parent module's direct children for Conv+BN sibling pairs.
    """
    children = list(parent.named_children())
    i = 0
    while i < len(children):
        name_cur, mod_cur = children[i]

        if (
            isinstance(mod_cur, nn.Conv2d)
            and i + 1 < len(children)
            and isinstance(children[i + 1][1], nn.BatchNorm2d)
        ):
            name_bn, mod_bn = children[i + 1]
            conv = mod_cur
            bn   = mod_bn

            std   = torch.sqrt(bn.running_var + bn.eps)
            scale = bn.weight / std  # gamma / std

            # Fuse weights: scale each output filter [C_out, C_in, kH, kW]
            conv.weight.data.mul_(scale.view(-1, 1, 1, 1))

            # Fuse bias: if conv had no bias, we create one to absorb the BN shift
            bias = (
                conv.bias.data
                if conv.bias is not None
                else torch.zeros(conv.out_channels,
                                  device=conv.weight.device,
                                  dtype=conv.weight.dtype)
            )
            conv.bias = nn.Parameter(scale * (bias - bn.running_mean) + bn.bias)

            # BN is now redundant — replace with Identity
            setattr(parent, name_bn, nn.Identity())
            print(f'  Fused: {name_cur} + {name_bn}')
            i += 2  # consumed both Conv and BN

        else:
            # Not a pair — recurse into this child's own subtree
            _fuse_bn_recursive(mod_cur)
            i += 1


def verify_bn_fusion(model: nn.Module) -> int:
    """Count remaining BatchNorm2d layers — should be 0 after fusion."""
    count = sum(1 for m in model.modules() if isinstance(m, nn.BatchNorm2d))
    print(f'Remaining BatchNorm2d layers after fusion: {count}')
    if count > 0:
        print('  Note: BN layers inside ViT blocks (GroupNorm/LayerNorm) are not BatchNorm2d')
    return count


# FP16 + channels_last + BN fusion
model_fused = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model_fused = model_fused.to(DEVICE).eval().half()
model_fused = model_fused.to(memory_format=torch.channels_last)

print('Fusing Conv+BN pairs...')
model_fused = fuse_model_bn(model_fused)
verify_bn_fusion(model_fused)

def infer_fused(x):
    with torch.no_grad():
        return model_fused(x)

result_fused = benchmark('4_fp16_nhwc_bn_fused', infer_fused,
                          input_fp16_nhwc, precision='fp16')
all_results.append(result_fused)
print(result_fused)

with torch.no_grad():
    out_fused = model_fused(input_fp16_nhwc).detach().clone()
measure_accuracy_degradation(reference_output, out_fused, 'fp16+nhwc+bn_fused')

Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


Fusing Conv+BN pairs...
Remaining BatchNorm2d layers after fusion: 0
[4_fp16_nhwc_bn_fused]
  Mean: 14.71ms | P50: 14.72ms | P95: 15.15ms | P99: 15.22ms
  FPS: 68.0 
  [fp16+nhwc+bn_fused] AbsRel: 0.0007 | Structural error: 0.0002


(0.00065979693, 0.00023292836)

In [8]:
import torch._dynamo

def compile_midas(model: nn.Module,
                   mode: str = 'reduce-overhead') -> nn.Module:
    """
    Compile MiDaS with a targeted graph break around the problematic
    ResNet stem that uses dynamic padding.

    Why the stem breaks compilation:
    ResNet patch_embed does: pad_h = ceil(H/2)*2 - H + 5
    This creates symbolic shapes that Inductor can't lower to CUDA kernels.
    Solution: disable compilation for just that submodule.
    """
    if (hasattr(model, 'pretrained') and
        hasattr(model.pretrained, 'model') and
        hasattr(model.pretrained.model, 'patch_embed')):

        stem = model.pretrained.model.patch_embed
        stem.forward = torch._dynamo.disable(stem.forward)
        print('Graph break inserted at: pretrained.model.patch_embed')
        print('Reason: dynamic padding in ResNet stem creates symbolic shapes')
        print('        that Inductor cannot lower to CUDA convolution kernels.')

    elif (hasattr(model, 'backbone') and
          hasattr(model.backbone, 'patch_embed')):
        stem = model.backbone.patch_embed.backbone.stem
        stem.forward = torch._dynamo.disable(stem.forward)
        print('Graph break inserted at: backbone.patch_embed.backbone.stem')

    compiled = torch.compile(
        model,
        backend   = 'inductor',
        mode      = mode,
        dynamic   = False,
        fullgraph = False,  # allow graph breaks
    )
    return compiled


# ── Stage 5: Everything + torch.compile ──
model_compiled = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid')
model_compiled = model_compiled.to(DEVICE).eval().half()
model_compiled = model_compiled.to(memory_format=torch.channels_last)
model_compiled = compile_midas(model_compiled)

def infer_compiled(x):
    with torch.no_grad():
        return model_compiled(x)

with torch.no_grad():
    _ = model_compiled(input_fp16_nhwc)
print('Compilation done.')

result_compiled = benchmark('5_compiled_fp16_nhwc', infer_compiled,
                              input_fp16_nhwc, precision='fp16')
all_results.append(result_compiled)
print(result_compiled)

with torch.no_grad():
    out_compiled = model_compiled(input_fp16_nhwc).detach().clone()
measure_accuracy_degradation(reference_output, out_compiled, 'compiled_fp16_nhwc')
save_depth_image(out_compiled, str(CONFIG['results_dir'] / '5_compiled.png'), 'Compiled FP16')

Using cache found in /home/RUS_CIP/st189432/.cache/torch/hub/intel-isl_MiDaS_master


Graph break inserted at: pretrained.model.patch_embed
Reason: dynamic padding in ResNet stem creates symbolic shapes
        that Inductor cannot lower to CUDA convolution kernels.
Compilation done.
[5_compiled_fp16_nhwc]
  Mean: 8.97ms | P50: 8.92ms | P95: 9.25ms | P99: 9.36ms
  FPS: 111.5 
  [compiled_fp16_nhwc] AbsRel: 0.0006 | Structural error: 0.0002


In [9]:
from torch.profiler import profile, ProfilerActivity, record_function

def run_profiler(name: str, infer_fn: Callable,
                  input_tensor: torch.Tensor,
                  warmup: int = 10, runs: int = 1) -> str:
    for _ in range(warmup):
        _ = infer_fn(input_tensor)
    torch.cuda.synchronize()

    trace_path = str(CONFIG['results_dir'] / f'1trace_{name}.json')

    with profile(
        activities    = [ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes = True,
        with_stack    = False
    ) as prof:
        for _ in range(runs):
            with record_function(name):
                _ = infer_fn(input_tensor)
            prof.step()
    prof.export_chrome_trace(trace_path)
    print(f'Chrome trace saved: {trace_path}')

    table = prof.key_averages().table(sort_by='cuda_time_total', row_limit=15)
    print(f'\n--- Profiler: {name} ---')
    print(table)
    return table

print('=' * 60)
print('PROFILING: Baseline FP32')
print('Look for: nchwToNhwc kernel, batch_norm kernel')
run_profiler('baseline_fp32',infer_fp32, input_fp32)

print('=' * 60)
print('PROFILING: FP16 + channels_last')
print('Expect: nchwToNhwc should disappear')
run_profiler('fp16_nhwc', infer_fp16_nhwc, input_fp16_nhwc)

print('=' * 60)
print('PROFILING: FP16 + channels_last + BN fused')
print('Expect: batch_norm should disappear')
run_profiler('fp16_nhwc_fused', infer_fused,  input_fp16_nhwc)

print('=' * 60)
print('PROFILING: Fully compiled')
print('Expect: fewer individual kernels, Triton fused kernels appear')
run_profiler('compiled',infer_compiled, input_fp16_nhwc)

PROFILING: Baseline FP32
Look for: nchwToNhwc kernel, batch_norm kernel
Chrome trace saved: results/1trace_baseline_fp32.json

--- Profiler: baseline_fp32 ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          baseline_fp32         0.00%       0.000us         0.00%       0.000us       0.000us      38.777ms       121.14%      38.777ms      38.777ms             1  
                                          baseli

'-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \n                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  \n-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  \n                                               compiled         0.60%      58.670us        20.38%       1.989ms       1.989ms       0.000us         0.00%       8.640ms       8.640ms             1  \n                             Torch-Compiled Region: 0/0         0.13%      12.363us        19.71%       1.924ms       1.924ms       0.000us         0.00%       8.640ms       8.640ms             1  \n   

In [ ]:
model = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid') #DPT_Small, further with int8 
model = model.eval()

dummy_input = torch.randn(1, 3, 384, 384)  # FP32,

torch.onnx.export(
    model,
    dummy_input,
    "midas_fp32.onnx",
    input_names=["input"],
    output_names=["depth"],
    opset_version=16,
    dynamic_axes={"input": {0: "batch"}, "depth": {0: "batch"}},
    do_constant_folding=True,
)

import onnx
from onnx.external_data_helper import convert_model_to_external_data, load_external_data_for_model
model_proto = onnx.load("midas_fp32.onnx", load_external_data=True)

onnx.save(
    model_proto,
    "midas_fp32_single.onnx",
    save_as_external_data=False 
)
onnx.checker.check_model("midas_fp32_single.onnx")

In [15]:
import onnxruntime as ort
import numpy as np
import cv2
import os

ONNX_PATH = "midas_fp32_single.onnx"
IMAGE_PATH = "Images/people.jpg"

print("Available providers:", ort.get_available_providers())

sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.intra_op_num_threads = 4

session = ort.InferenceSession(
    ONNX_PATH,
    sess_opts,
    providers=["CPUExecutionProvider"]  # CPU only
)

inp  = session.get_inputs()[0]
out  = session.get_outputs()[0]
print(f"Input:  {inp.name}  shape={inp.shape}  dtype={inp.type}")
print(f"Output: {out.name}  shape={out.shape}  dtype={out.type}")

frame = cv2.imread(IMAGE_PATH)
assert frame is not None, f"Could not read {IMAGE_PATH}"
orig_size = frame.shape[:2]

resized = cv2.resize(frame, (384, 384))
rgb     = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
tensor  = rgb.transpose(2, 0, 1)[np.newaxis, ...]

print(f"Input tensor: shape={tensor.shape} dtype={tensor.dtype} "
      f"min={tensor.min():.3f} max={tensor.max():.3f}")

import time

for _ in range(3):
    session.run(None, {inp.name: tensor})

times = []
for _ in range(10):
    t0 = time.perf_counter()
    outputs = session.run(None, {inp.name: tensor})
    t1 = time.perf_counter()
    times.append((t1 - t0) * 1000)

depth = outputs[0]
print(f"Output shape: {depth.shape}  dtype={depth.dtype}")
print(f"Depth range:  min={depth.min():.3f}  max={depth.max():.3f}")
print(f"Mean latency (CPU): {np.mean(times):.1f} ms")

assert not np.isnan(depth).any(),  "NaN values in output — model is broken"
assert depth.max() > depth.min(),  "Flat output — model is broken"
fps = 1000.0/np.mean(times)
d = depth.squeeze()
d_norm = (d - d.min()) / (d.max() - d.min() + 1e-6)
d_uint8 = (d_norm * 255).astype(np.uint8)
colored = cv2.applyColorMap(d_uint8, cv2.COLORMAP_INFERNO)
cv2.putText(colored,
        f"{fps:.0f} FPS",
        (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
        (200, 200, 200), 2, cv2.LINE_AA)
colored = cv2.resize(colored, (orig_size[1], orig_size[0]))
cv2.imwrite("depth_onnx_fp32.png", colored)


Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
Input:  input  shape=['batch', 3, 384, 384]  dtype=tensor(float)
Output: depth  shape=[1, 384, 384]  dtype=tensor(float)
Input tensor: shape=(1, 3, 384, 384) dtype=float32 min=0.000 max=1.000
Output shape: (1, 384, 384)  dtype=float32
Depth range:  min=87.678  max=1898.613
Mean latency (CPU): 822.2 ms


True